Implementazione di un encoder Transformer con PyTorch

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """Codifica posizionale sinusoidale (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Calcolo della matrice PE
        pe = torch.zeros(max_len, d_model) # (L, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        # div_term = e^(arange(0, d_model, 2) * -(log(10000.0) / d_model))
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term) # dimensioni pari
        pe[:, 1::2] = torch.cos(position * div_term) # dimensioni dispari

        pe = pe.unsqueeze(0) # (1, L, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class TransformerEncoder(nn.Module):
    """
    Stack di N strati encoder con multi-head self-attention.
    Cfr. nn.TransformerEncoderLayer:
    docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html
    """

    def __init__(self, vocab_size: int, d_model: int = 512,
                 nhead: int = 8, num_layers: int = 6,
                 dim_feedforward: int = 2048,
                 dropout: float = 0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # PyTorch fornisce un'implementazione ottimizzata dell'encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True # input: (batch, seq, feature)
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )

        self.d_model = d_model

    def forward(self, src: torch.Tensor,
                src_key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        # src: (batch, seq_len) indici interi dei token
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        # Mascheratura dei token di padding (True = ignora)
        out = self.encoder(x, src_key_padding_mask=src_key_padding_mask)

        return out # (batch, seq_len, d_model)


# --- Esempio di utilizzo ---
if __name__ == "__main__":
    VOCAB = 10000
    BATCH = 32
    SEQ_LEN = 64

    model = TransformerEncoder(
        vocab_size=VOCAB, d_model=512,
        nhead=8, num_layers=6
    )

    tokens = torch.randint(0, VOCAB, (BATCH, SEQ_LEN))
    output = model(tokens)

    print(output.shape) # torch.Size([32, 64, 512])
